In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
products = spark.read.table("db_ws.bronze.crm_prd")
display(products)

In [0]:
# Filling nulls for HL Road Frame with its average prd_cost of HL Road Frame
hl_road_frame_avg = products.filter(col("prd_nm").startswith("HL Road Frame")).agg(avg("prd_cost")).collect()[0][0]
products = products.withColumn("prd_cost",when(col("prd_nm").startswith("HL Road Frame") & col("prd_cost").isNull(), hl_road_frame_avg).otherwise(col("prd_cost"))).withColumn("prd_line",when(col("prd_line").isNull(),lit("Unkowen")).otherwise(col("prd_line")))
display(products)

In [0]:
products.write.format("delta").saveAsTable("db_ws.silver.crm_prd")